# T4 GRPO Demo — Bioresearch (Colab Free, Qwen2.5-1.5B, All 14 Tasks)

<a href="https://colab.research.google.com/drive/1UIJTvMm2cWBpM46FI_MFEEthvN7JS-Q0#scrollTo=f2v4Y85v1Zna" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

End-to-end GRPO training run sized for a **Colab Free T4 (16 GB, Turing sm_75)** that still trains and evaluates **all 14 bioresearch tasks** in a single sitting.

All heavy logic lives in [`training_core.py`](../training_core.py) and [`training_a100.py`](../training_a100.py) — both are GPU-agnostic, so we reuse them verbatim. This notebook is a thin driver.

---

## Why Qwen2.5-1.5B and not 3B/7B on a 16 GB Turing card

We hard-pin Qwen2.5-1.5B-Instruct (4-bit) and spend the remaining T4 headroom on running all 14 tasks instead of a bigger model:

- 1.5B + LoRA r=16 + 4-bit + num_gen=4 KV cache fits in **~5-6 GB**. That leaves **~10 GB** for the lab-task rollout generates inside the reward function (each lab task's reward fn does up to `TRAIN_LAB_MAX_STEPS=2` extra `model.generate` calls per completion).
- 3B at the same settings fits but **doubles wall-clock** (~140-180 min for the full run) and pushes us past Colab Free's ~90 min.
- **GRPO teaches *behavior*, not knowledge.** The reward function gates on valid JSON, correct phase transitions, and right tool calls — 1.5B has enough biology priors for the format-bound tasks even without a bigger model.
- **Honest caveat:** knowledge-bound tasks (`protein_function`, `kegg_pathway_reasoning`) will show smaller deltas on 1.5B than on 7B because GRPO can't teach facts the base model doesn't already encode. The notebook calls this out in Cell 25 so judges read flat curves on those two tasks correctly.

**Net Colab Free T4 wall-clock for the full 14-task demo run:** ~75-90 min train + ~10 min eval = **~85-100 min total**.

**Hackathon checklist** (all artifacts produced by this notebook):

- [x] Live Trackio dashboard (Cell 8) — `bioresearch-grpo-t4` project, optional HF Space sync
- [x] Reward-distribution diagnostic + boxplot (Cell 18)
- [x] Baseline rollouts on paired task_ids (Cell 16)
- [x] GRPO training run with mid-run LoRA checkpoints every 20 steps (Cells 20-22)
- [x] Trained rollouts on the same paired task_ids (Cell 24)
- [x] Before-vs-after table + bar chart (Cell 26)
- [x] Sample rollout transcripts side-by-side (Cell 28)
- [x] Per-task reward curves (Cell 30) — static fallback in case Trackio is offline
- [x] Saved LoRA + optional HF Hub push (Cell 32)


## 1 · Install dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# T4 = Turing (sm_75) = fp16 only, no bf16, no flash-attn 2. Unsloth detects this
# and falls back to xformers automatically. xformers <0.0.28 still ships sm_75
# wheels, so the same pin used in train_grpo_l40s.ipynb (sm_89) works here too.
# trl: GRPOConfig/GRPOTrainer landed in 0.13.0 (Dec 2024); processing_class= and
# vllm_mode= matured in 0.15+. We pin >=0.15,<1.0 so pip resolves to ~0.18.x
# (the last mature 0.x line) and avoids the 1.x reorg. The legacy "trl<0.12" pin
# in train_grpo_a100.ipynb / train_grpo_colab.ipynb is a vestige from when Unsloth
# shipped GRPO via its own fork — it now ImportErrors on `from trl import GRPOConfig`.
!pip install -q --no-deps "xformers<0.0.28" "trl>=0.15,<1.0" peft accelerate bitsandbytes
!pip install -q httpx fastapi uvicorn pydantic pandas matplotlib datasets scipy
!pip install -q trackio  # live dashboard; the pipeline still runs without it

# vLLM is intentionally NOT installed on Colab T4 (or any Colab/Modal Jupyter):
#   - vLLM has no sm_75 wheels for its custom kernels — would need source build.
#   - Full install (~500 MB of wheels including ray, opencv-python, mistral-common,
#     and a custom torch build) frequently clobbers Unsloth's pinned torch/xformers.
#   - --no-deps install drops vLLM module files without its runtime deps, which
#     then triggers a `ModuleNotFoundError: cv2` deep inside Unsloth's startup hook
#     `fix_vllm_guided_decoding_params()` -> `import vllm`.
# Wall-clock without vLLM on T4 is ~75-90 min for the full 14-task demo run.
# `num_generations=4` in Cell 20 is what tightens the reward curves; vLLM was never
# going to help us here.

# IMPORTANT: do this LAST so nothing above re-upgrades it.
# `transformers` in the unsloth/trl stack pins `huggingface_hub>=0.34,<1.0`.
# Many Colab runtimes ship `huggingface_hub==1.12.0` preinstalled, which breaks
# `from unsloth import FastLanguageModel` at the dep-version check.
# Force a compatible version with --force-reinstall so the existing 1.x is replaced.
!pip install -q --force-reinstall --no-deps "huggingface_hub>=0.34,<1.0"

# After this cell finishes, RESTART THE RUNTIME ONCE, then re-run from Cell 1.
# Reason: if `huggingface_hub` (or a partially-installed `vllm`) was already
# imported by the runtime before these fixes ran, Python has cached the broken
# module objects and the new installs won't take effect until the kernel restarts.
# (On a fresh Colab session this is a no-op.)
print("\nIf this is your FIRST run on a fresh runtime: continue normally.")
print("Otherwise: Runtime -> Restart, then re-run from Cell 1.")

## 2 · Clone the environment repo (Colab)

In [ ]:
import os, subprocess, sys

REPO_URL = os.environ.get(
    "BIORESEARCH_REPO_URL",
    "https://huggingface.co/spaces/anirudhchida/bioresearch",
)

if not os.path.isdir("bioresearch"):
    if "anirudhchida" in REPO_URL:
        raise RuntimeError(
            "Set BIORESEARCH_REPO_URL (or edit REPO_URL above) to your fork — "
            "the placeholder URL will 404 on clone."
        )
    subprocess.run(["git", "clone", REPO_URL, "bioresearch"], check=True)

sys.path.insert(0, "/content/bioresearch")
print("Setup complete")

## 3 · Boot the OpenEnv server

The reward loop talks HTTP to this subprocess. Current OpenEnv releases expose `/health`, `/schema`, `/docs`, and `/openapi.json` — the legacy `/info` is gone.

In [ ]:
import subprocess, time, httpxe

server_proc = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/bioresearch",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

for _ in range(40):
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=1.0).status_code == 200:
            print("Server up")
            break
    except Exception:
        time.sleep(1.0)
else:
    raise RuntimeError("OpenEnv server failed to start")

## 4 · Wire up `training_core` + Trackio

We reuse [`training_a100.py`](../training_a100.py) verbatim — every helper there (`setup_trackio`, `collect_rollouts_with_pool`, `reward_distribution_diagnostic`, `before_after_table`, `render_sample_transcripts`, `make_lab_shaping_callback`) is GPU-agnostic.

T4-specific tightening: we cap `TRAIN_LAB_MAX_STEPS=2` (vs L40s's 6, default 4) because each lab task in the reward fn does up to that many extra `model.generate` calls per completion. With `num_generations=4` and ~57% of prompts being lab tasks, this single knob saves us ~25-35 min over the run.

Trackio init is wrapped — if the install in Cell 1 failed or the network blocks the HF Space, `setup_trackio` degrades gracefully to `report_to="none"` and the rest of the notebook still produces all the same artifacts (just no live dashboard). Set `HF_SPACE_ID` to a `username/space-name` you own to enable persistent sync.

In [ ]:
import os
import training_core
import training_a100
from training_core import (
    LEGACY_TASKS, LAB_TASKS, ALL_TASKS, DEFAULT_T4_TASKS,
    TRAIN_LAB_MAX_STEPS, EVAL_LAB_MAX_STEPS,
)

# T4 lab rollout budget: half the default. Each lab task in the reward function
# does up to TRAIN_LAB_MAX_STEPS extra model.generate() calls per completion --
# at num_generations=4 with ~57% lab tasks per step that compounds fast.
# 2 keeps process-reward density nonzero while saving ~25-35 min over the run.
os.environ["TRAIN_LAB_MAX_STEPS"] = "2"
training_core.TRAIN_LAB_MAX_STEPS = 2
training_core.EVAL_LAB_MAX_STEPS = 8

SERVER_URL = "http://127.0.0.1:8000"
training_core.configure_env(SERVER_URL)

# Optional persistent Trackio dashboard. Leave HF_SPACE_ID unset for a
# local-only dashboard (still visible in Colab via the trackio CLI).
trackio_cfg = training_a100.setup_trackio(
    project="bioresearch-grpo-t4",
    run_name="qwen2p5-1p5b-t4-80steps",
    space_id=os.environ.get("HF_SPACE_ID"),
    config={"model": "Qwen2.5-1.5B-Instruct-bnb-4bit", "lora_r": 16, "num_generations": 4},
)
print("Trackio config:", trackio_cfg)

probe = training_core.env_reset(task_type="dna_classification").observation
print(f"Probe OK  task_id={probe.task_id}  question[:60]={probe.question[:60]!r}")

## 5 · Choose the task list

Default trains all 14 tasks — this is the demo run. The 6-task T4 subset and a custom subset are documented as commented lines.

In [ ]:
TASK_LIST = ALL_TASKS                       # 14 reward curves — full demo
# TASK_LIST = DEFAULT_T4_TASKS              # 6-task fast subset for smoke runs
# TASK_LIST = ["dna_reasoning", "clinical_diagnosis", "protein_hypothesis_lab"]

print("Training on:", TASK_LIST)
print("  legacy:", [t for t in TASK_LIST if t in LEGACY_TASKS])
print("  lab   :", [t for t in TASK_LIST if t in LAB_TASKS])

## 6 · Build the mixed-task dataset

`n_per_task=4` (56 rows for all 14 tasks). Smaller than L40s's 8 because each prompt is more expensive on T4 — 56 rows × 4 generations × 80 steps gives ~5.7 epochs, which is in the same range as the L40s notebook (~5.4 epochs) where curves visibly stabilise.

In [ ]:
N_PER_TASK = 4
train_ds = training_core.build_mixed_dataset(TASK_LIST, n_per_task=N_PER_TASK, seed=42)
print(train_ds)
print("Sample row keys:", train_ds.column_names)

## 7 · Load Qwen2.5-1.5B + LoRA (r=16)

Hard-pinned to **Qwen2.5-1.5B-Instruct-bnb-4bit**. See the rationale in Cell 0 — 3B doubles wall-clock past Colab Free's idle-disconnect window, and 7B OOMs once lab rollouts kick in.

After loading we print VRAM headroom and assert >8 GB free. Tighter budget than the L40s notebook because Colab Free hands out variable T4 SKUs (some with ~14 GB usable, some ~15 GB) — but if the assertion trips, something has changed in the deps stack and we want to know loudly before training.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"        # fixed for T4 demo
# MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"        # only on Colab Pro / longer sessions; doubles wall-clock
MAX_SEQ_LEN = 2048                                            # 3072 on L40s -> 2048 on T4 (~33% KV cache cut)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                                     # 32 on L40s -> 16 on T4 (~half adapter VRAM)
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
tokenizer.padding_side = "left"
training_core.configure_model(model, tokenizer, max_new_tokens=256)   # 320 on L40s -> 256 on T4

# Suppress HF's "Both max_new_tokens and max_length are set" warning at the source.
# Qwen2.5's generation_config.json ships with max_length=32768; we always pass
# max_new_tokens explicitly (256 here, plus TRL's max_completion_length=256 in
# Cell 20). Dropping max_length from the model-level config means HF won't warn
# from ANY generate path - _generate_once (lab rollouts), TRL's GRPO sampler
# (~16 generates/step), or trainer.evaluate. Without this fix the warning spams
# stdout ~1300+ times over an 80-step run, drowning the actual training logs.
if model.generation_config is not None:
    model.generation_config.max_length = None

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM headroom after model load: {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total")
    assert free > 8 * 1e9, (
        f"Only {free / 1e9:.1f} GB free after loading 1.5B on T4 - expected >8 GB. "
        "Something in the deps stack regressed; check 4-bit loading + LoRA r=16."
    )
print("Model + tokenizer loaded and registered with training_core")

## 8 · Baseline rollouts (BEFORE training)

Run the *untrained* policy on a fixed seed of 3 held-out `task_id`s per task. We capture the pool here and reuse it after training so the before/after compare on identical briefs.

L40s used n=5 per task; T4 uses n=3 to cut ~6-8 min off baseline collection (each rollout costs ~8-12s on T4 with the lab task multi-step generates). Per-task means at n=3 still show clear directional signal in the table.

In [ ]:
FastLanguageModel.for_inference(model)        # 2x faster greedy-ish gen

baseline_rows, eval_pool = training_a100.collect_rollouts_with_pool(
    TASK_LIST, n_per_task=3, seed=2026,
)
training_a100.save_rollouts(baseline_rows, "baseline_rollouts.json")

# Per-task summary so we have a number to point at on the slide.
import statistics
print(f"Baseline rollouts (n={len(baseline_rows)}):")
for task in TASK_LIST:
    rs = [r["reward"] for r in baseline_rows if r["task_type"] == task]
    if rs:
        print(f"  {task:30s}  mean={statistics.fmean(rs):.3f}  n={len(rs)}")

FastLanguageModel.for_training(model)         # back to training mode

## 9 · Reward-distribution diagnostic

Score a hand-curated bag of completions per task through the reward function. The point isn't to evaluate the model — the completions are canned. The point is to **prove to the judges** that the reward signal:

1. Spans a meaningful range (not all 0 or 1).
2. Reflects completion quality (correct answers get high reward, garbage gets low).
3. Has variance for GRPO's relative advantage to actually optimize against.

In [ ]:
import matplotlib.pyplot as plt

diag_df = training_a100.reward_distribution_diagnostic(TASK_LIST, n_samples_per_task=4)
print(diag_df)

fig, ax = plt.subplots(figsize=(12, 4))
tasks_in_order = [t for t in TASK_LIST if t in set(diag_df["task_type"])]
data = [diag_df[diag_df["task_type"] == t]["reward"].tolist() for t in tasks_in_order]
ax.boxplot(data, labels=tasks_in_order, vert=True, showmeans=True)
ax.set_ylabel("Reward (canned-completion bag)")
ax.set_title("Reward signal has variance — GRPO has something to optimize")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("reward_diagnostic.png", dpi=140)
plt.show()

## 10 · `GRPOConfig` + `GRPOTrainer` (Trackio-wired)

T4 defaults: `num_generations=4`, `max_completion_length=256`, `max_steps=80`, `save_steps=20`, `fp16=True` (forced — Turing has no bf16). These are tuned in four independent dimensions:

- **`num_generations=4`** (vs L40s's 10): GRPO advantage variance scales `1/num_generations`, so 10 -> 4 widens noise ~58% — but at 4 we still have a meaningful relative-advantage signal, and KV cache for 4 generations on a 1.5B model leaves ~10 GB free for lab rollouts. Going lower would collapse GRPO into an unweighted REINFORCE-like update.
- **`max_completion_length=256`** (vs L40s's 640): ~60% generation-time cut. Qwen 1.5B's JSON action+reasoning fits comfortably under 256 tokens for this env's prompt format.
- **`max_steps=80`** (vs L40s's 150): ~5.7 epochs at n_per_task=4. Reward curves on a 1.5B model plateau earlier than on 7B because the policay's action space is smaller — extending past 80 mostly burns wall-clock without measurable lift.
- **`save_steps=20`** (vs L40s's 50): writes the LoRA every ~15-20 min so a Colab Free idle-disconnect mid-train does not lose everything. Cheap (~50 MB checkpoint).

**`fp16=True` is hardcoded** — Turing sm_75 has no bf16 support. We do NOT use `_BF16_OK` autodetect here because the answer is always False on T4 and we want the cell to fail loudly if someone runs it on the wrong card.

**`USE_VLLM`** auto-detects vLLM and enables colocated rollouts. On Colab T4 we deliberately skip vLLM in Cell 1 (no sm_75 wheels + Unsloth import-hook crash on partial install), so this will normally print "vLLM not available" and use native HF generation.

**`LabShapingCallback`** drains the per-step lab rollout log on every `on_log` event so the dashboard shows process vs terminal reward decomposition for each lab task.

In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer

reward_fns = [training_core.make_reward_fn(t) for t in TASK_LIST]
print("Reward functions wired:")
for fn in reward_fns:
    print(" -", fn.__name__)

# T4 = Turing = no bf16. Hardcode fp16=True regardless of detection.
USE_VLLM = False
try:
    import vllm  # noqa: F401
    USE_VLLM = True
    print("vLLM detected -> enabling fast colocated rollouts (unexpected on T4)")
except Exception:
    print("vLLM not available -> using native HF generation (expected on Colab T4)")

_grpo_kwargs = dict(
    output_dir="grpo_bioresearch_t4",
    learning_rate=3e-6,
    per_device_train_batch_size=1,         # 2 on L40s -> 1 on T4 (VRAM)
    gradient_accumulation_steps=4,         # 2 on L40s -> 4 on T4 (keep 4 prompts/step effective batch)
    num_generations=4,                     # 10 on L40s -> 4 on T4 (KV cache + wall-clock)
    max_prompt_length=1024,                # 1280 on L40s -> 1024 on T4
    max_completion_length=256,             # 640 on L40s -> 256 on T4 (~60% generation-time cut)
    max_steps=80,                          # 150 on L40s -> 80 on T4 (~5.7 epochs at n_per_task=4)
    logging_steps=1,
    save_strategy="steps",
    save_steps=20,                         # 50 on L40s -> 20 on T4 (resilient to disconnect)
    bf16=False,                            # Turing has no bf16 — never enable on T4
    fp16=True,                             # forced — overrides the L40s _BF16_OK autodetect
    beta=0.04,
    max_grad_norm=1.0,
    seed=42,
    use_vllm=USE_VLLM,
    **trackio_cfg,
)
if USE_VLLM:
    _grpo_kwargs["vllm_mode"] = "colocate"  # single-GPU T4

# Forward-compat: older TRL versions don't know `vllm_mode`, newer don't know `use_vllm`.
# Strip unknowns until GRPOConfig accepts the dict.
while True:
    try:
        grpo_config = GRPOConfig(**_grpo_kwargs)
        break
    except TypeError as exc:
        msg = str(exc)
        dropped = False
        for k in ("vllm_mode", "use_vllm"):
            if k in _grpo_kwargs and k in msg:
                print(f"[GRPOConfig] dropping unsupported kwarg {k!r} ({msg.splitlines()[0]})")
                _grpo_kwargs.pop(k)
                dropped = True
                break
        if not dropped:
            raise

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=train_ds,
    reward_funcs=reward_fns,
    callbacks=[training_a100.make_lab_shaping_callback()],
)
print("Trainer ready")

## 11 · Train

Expected wall-clock on Colab Free T4: **~75-90 min train + ~10 min eval = ~85-100 min total** for the full 14-task run.

If Colab disconnects mid-train, the LoRA checkpoint at `grpo_bioresearch_t4/checkpoint-XX/` (last multiple of 20 before disconnect) is the recovery point — load it back with `model.load_adapter(...)` in a fresh runtime and re-run from Cell 23 to skip retraining.

In [ ]:
trainer.train()
training_a100.trackio_finish()  # noop if Trackio is not in use

## 12 · Trained rollouts (AFTER training)

Same `task_ids` as Cell 16 — `eval_pool` pins them. This is what makes the before/after table a *paired* comparison instead of two unpaired draws. We use n=3 here (matching baseline) so every trained row has a baseline counterpart.

In [ ]:
FastLanguageModel.for_inference(model)

trained_rows = training_a100.collect_rollouts(
    TASK_LIST, n_per_task=3, seed=2026, task_id_pool=eval_pool,
)
training_a100.save_rollouts(trained_rows, "trained_rollouts.json")

import statistics
print(f"Trained rollouts (n={len(trained_rows)}):")
for task in TASK_LIST:
    rs = [r["reward"] for r in trained_rows if r["task_type"] == task]
    if rs:
        print(f"  {task:30s}  mean={statistics.fmean(rs):.3f}  n={len(rs)}")

## 13 · Before-vs-after table + bar chart

Headline judging artifact #1. Per-task delta in mean reward, sorted by biggest win first. The bar chart makes the wins obvious at a glance.

**Reading the table on a 1.5B model:** GRPO teaches *behavior* (JSON discipline, phase transitions, tool-call selection), not *facts*. On a 1.5B base model, knowledge-bound tasks like `protein_function` and `kegg_pathway_reasoning` may show small or even zero deltas — that is the model's knowledge ceiling, not a training-budget problem. Format-bound and procedure-bound tasks (`dna_classification`, `evidence_ranking`, `clinical_diagnosis`, all the `*_lab` tasks) are where the GRPO signal compounds, and that is where the deltas should be visible. The L40s notebook with Qwen 7B closes most of that knowledge gap; if you have the budget, run that one alongside this for the full picture.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

table = training_a100.before_after_table(baseline_rows, trained_rows)
print(table.to_string(index=False) if hasattr(table, "to_string") else table)

if hasattr(table, "iloc"):
    tasks = table["task_type"].tolist()
    deltas = table["delta"].tolist()
    baselines = table["baseline_mean"].tolist()
    traineds = table["trained_mean"].tolist()
else:
    tasks = [r["task_type"] for r in table]
    deltas = [r["delta"] for r in table]
    baselines = [r["baseline_mean"] for r in table]
    traineds = [r["trained_mean"] for r in table]

x = np.arange(len(tasks))
fig, ax = plt.subplots(figsize=(13, 5))
width = 0.38
ax.bar(x - width/2, baselines, width, label="Baseline (untrained)")
ax.bar(x + width/2, traineds, width, label="After GRPO training")
for i, d in enumerate(deltas):
    color = "green" if d > 0 else ("red" if d < 0 else "gray")
    ax.annotate(f"{d:+.2f}", xy=(x[i], max(baselines[i], traineds[i]) + 0.02),
                ha="center", color=color, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(tasks, rotation=40, ha="right")
ax.set_ylabel("Mean reward")
ax.set_title("Before vs After GRPO — per-task mean reward (paired task_ids, L40s + Qwen2.5-7B)")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("before_after_bar.png", dpi=140)
plt.show()

## 14 · Sample rollout transcripts (the headline demo artifact)

Headline judging artifact #2. For each task, the 2 `task_id`s with the largest positive delta are shown side-by-side: prompt → baseline completion + reward → trained completion + reward. Judges can literally read the qualitative jump.

In [ ]:
from IPython.display import Markdown, display

transcripts_md = training_a100.render_sample_transcripts(
    baseline_rows, trained_rows, k=2, max_chars_per_block=600,
)
with open("sample_transcripts.md", "w") as f:
    f.write(transcripts_md)
print(f"Wrote sample_transcripts.md ({len(transcripts_md)} chars)")

display(Markdown(transcripts_md))

## 15 · Per-task reward curves (from `trainer.state.log_history`)

Live Trackio is the primary instrument; this is the static fallback so the artifact survives the runtime ending. Each curve is a single task's reward function — `0.0` rows (rows from other tasks) are masked before smoothing.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_df = pd.DataFrame(trainer.state.log_history)
print("Reward-related columns:", [c for c in log_df.columns if "reward" in c])

fig, ax = plt.subplots(figsize=(12, 5))
for task in TASK_LIST:
    candidates = [c for c in log_df.columns
                  if task in c and "reward" in c and "std" not in c.lower() and "process" not in c and "terminal" not in c]
    if not candidates:
        continue
    col = candidates[0]
    subset = log_df[["step", col]].dropna()
    subset = subset[subset[col] > 0.0].reset_index(drop=True)
    if subset.empty:
        continue
    smooth = subset[col].rolling(window=10, min_periods=1).mean()
    ax.plot(subset["step"], smooth, linewidth=2.0, label=task)

ax.set_xlabel("GRPO step")
ax.set_ylabel("Reward (10-step EMA, gated rows only)")
ax.set_title("Per-task reward curves — T4 demo run (Qwen2.5-1.5B, num_generations=4)")
ax.legend(loc="best", fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=140)
plt.show()

## 16 · Save LoRA + (optional) Hub push + teardown

In [ ]:
model.save_pretrained("grpo_bioresearch_t4_lora")
tokenizer.save_pretrained("grpo_bioresearch_t4_lora")
print("LoRA saved to grpo_bioresearch_t4_lora/  (~50 MB checkpoint - easy to share)")

# Optional: push to the Hub if HF_TOKEN is set in the env. The 1.5B + r=16 LoRA
# weighs in around 50 MB so the push is fast and the resulting repo is cheap to
# pull from for inference notebooks (see train_grpo_inference_mirror.ipynb).
HF_REPO = os.environ.get("HF_PUSH_REPO")
if HF_REPO and os.environ.get("HF_TOKEN"):
    try:
        model.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
        tokenizer.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
        print(f"Pushed to https://huggingface.co/{HF_REPO}")
    except Exception as e:
        print(f"Hub push failed ({type(e).__name__}: {e}); LoRA still saved locally.")
else:
    print("Skipped Hub push (set HF_PUSH_REPO + HF_TOKEN to enable).")

server_proc.terminate()
try:
    server_proc.wait(timeout=5)
except Exception:
    server_proc.kill()
print("Server terminated")